# Model 1 item-text encoding and SASRec ablation
Written by: Abhinita Sanabada

Start with an **encoder smoke run**. Candidate model IDs are shown for editing;
they are not recorded as team-approved choices. This notebook does not generate
user profiles or run the full profile-distilled Model 1. It never evaluates test
metrics. The Drive API reads the supplied folder directly, so no shortcut path
is required. Use a Google account with access to the shared data folder.


In [ ]:
from pathlib import Path
REPO_URL = "https://github.com/Ashley96yy/data298a-team5-topic22.git"
REPO_REF = "abhinita/item-text-fusion"  # replace with reviewed commit for final experiments
PROJECT = Path("/content/298A_item_text")
PROCESSED_FOLDER_ID = "1OZFOjwoZjgB2_JR9u2fkHxRBcZnsWm5G"
ENCODER = "minilm"  # "minilm" or "blair"
MODEL_IDS = {"minilm": "sentence-transformers/all-MiniLM-L6-v2",
             "blair": "hyp1231/blair-roberta-base"}
MODEL_ID = MODEL_IDS[ENCODER]  # edit after the team confirms exact models
MODEL_REVISION = None  # first execution resolves and records an immutable commit
ENCODE_MODE = "smoke"  # "smoke": 16 items; "full": entire catalog
TRAIN_MODE = "off"     # "off", "smoke" (128 users/2 epochs), or "full"
GRID_INDEX = 7         # fixed Model 0 winner for an item-text ablation; not a Model 1 search
SEED = 42
BATCH_SIZE = 32


## Load the reviewed source and optional dependencies
Use a GPU runtime for full encoding/training. Installation can take several minutes.


In [ ]:
import subprocess, sys, os
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", SOURCE_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-item-text.txt"], check=True)


## Authenticate and copy verified frozen inputs
This may prompt you to sign in. Only fixed experiment input files are downloaded.
Outputs are saved in your My Drive under `298A/model1_item_text`.


In [ ]:
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import hashlib, json, re

auth.authenticate_user()
drive.mount("/content/drive")
credentials, _ = default()
service = build("drive", "v3", credentials=credentials, cache_discovery=False)
if not re.fullmatch(r"[A-Za-z0-9_-]+", PROCESSED_FOLDER_ID):
    raise ValueError("Invalid processed folder ID")
processed = PROJECT / "data/processed"
processed.mkdir(parents=True, exist_ok=True)
children_cache = {}
def child(folder_id, name):
    if folder_id not in children_cache:
        files, token = [], None
        while True:
            response = service.files().list(q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken,files(id,name,mimeType)", pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            files.extend(response.get("files", []))
            token = response.get("nextPageToken")
            if not token: break
        children_cache[folder_id] = files
    matches = [f for f in children_cache[folder_id] if f['name'] == name]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {name!r} in folder {folder_id}; found {len(matches)}")
    return matches[0]['id']
def download(relative, expected=None):
    destination = processed / relative
    if destination.exists() and expected and hashlib.sha256(destination.read_bytes()).hexdigest() == expected:
        return
    file_id = PROCESSED_FOLDER_ID
    for name in Path(relative).parts:
        file_id = child(file_id, name)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".download")
    with temporary.open("wb") as output:
        downloader = MediaIoBaseDownload(output, service.files().get_media(fileId=file_id, supportsAllDrives=True))
        done = False
        while not done:
            _, done = downloader.next_chunk()
    if expected and hashlib.sha256(temporary.read_bytes()).hexdigest() != expected:
        temporary.unlink()
        raise ValueError(f"Frozen checksum mismatch: {relative}")
    temporary.replace(destination)
download("manifest.json")
manifest = json.loads((processed / "manifest.json").read_text())
required = ["dataset_info.json", "item_text.parquet"]
if TRAIN_MODE != "off":
    required += ["rec_eval.py", "sequences.parquet", "eval/eval_protocol.json", "eval/neg_val.npy",
                 "eval/neg_test.npy", "eval/eval_cases.parquet", "groups/item_groups.parquet", "groups/user_groups.parquet"]
for name in required:
    download(name, manifest["processed_files"][name]["sha256"])
print("Verified files:", required)


## Encode items
Smoke artifacts are explicitly marked partial and cannot be used for training.
After the smoke succeeds and the team confirms the encoder, change `ENCODE_MODE`
to `full` and rerun settings and subsequent cells. Each run uses a new output folder.


In [ ]:
import torch
from datetime import datetime, timezone
from huggingface_hub import HfApi
if ENCODE_MODE not in ("smoke", "full") or TRAIN_MODE not in ("off", "smoke", "full"):
    raise ValueError("Invalid run mode")
if TRAIN_MODE != "off" and ENCODE_MODE != "full":
    raise ValueError("Training needs full-catalog embeddings, even for a training smoke test")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if (ENCODE_MODE == "full" or TRAIN_MODE == "full") and DEVICE != "cuda":
    raise RuntimeError("Choose a Colab GPU runtime for this full run")
if MODEL_REVISION is None:
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
print("Encoder:", MODEL_ID, "revision:", MODEL_REVISION, "device:", DEVICE)
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = Path("/content/drive/MyDrive/298A/model1_item_text") / f"{ENCODER}_{ENCODE_MODE}_{STAMP}"
command = [sys.executable, "-m", "src.data.item_embeddings", "--processed", str(processed),
    "--encoder", ENCODER, "--model-id", MODEL_ID, "--revision", MODEL_REVISION,
    "--device", DEVICE, "--batch-size", str(BATCH_SIZE), "--output-dir", str(OUTPUT)]
if ENCODE_MODE == "smoke": command += ["--limit-items", "16"]
subprocess.run(command, check=True)
metadata = json.loads((OUTPUT / "metadata.json").read_text())
print(json.dumps(metadata, indent=2))
print("Saved:", OUTPUT)


## Optional item-only training
This uses the shared trainer and frozen validation NDCG@10. It is not the full
profile-distilled experiment. A full run requires agreed encoder settings.
The best checkpoint includes the frozen item-text matrix and learned projection.


In [ ]:
if TRAIN_MODE == "off":
    print("Training disabled; encoder smoke only.")
else:
    RUN = OUTPUT / f"training_seed_{SEED}_{TRAIN_MODE}"
    command = [sys.executable, "-m", "src.training.train_item_text", "--config", "configs/config.json",
        "--grid-index", str(GRID_INDEX), "--seed", str(SEED), "--embeddings", str(OUTPUT),
        "--output-dir", str(RUN), "--device", DEVICE]
    if TRAIN_MODE == "smoke": command += ["--smoke"]
    subprocess.run(command, check=True)
    print(json.dumps(json.loads((RUN / "summary.json").read_text()), indent=2))
    print("Saved training run:", RUN)
